In [1]:
import numpy as np
import pandas as pd
from pytomography.io.SPECT import simind

import torch
import nibabel as nib
import sys
import os
sys.path.append("C:\\ac225\\Ac-225")
import functions
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy.ndimage import gaussian_filter
from skimage.metrics import structural_similarity as ssim

In [2]:
true_lk= {"p24": 57.1, "p25":95.2, "p33":59.0 ,"p15":68.8, "p23":88.3, "p17":23.5, "p32":25.9, "p28":47.8, "p31":53.8, "p13":40.5, "p12":49.0, "p11": 52.8016}
true_rk= {"p24": 60.3, "p25":116.1, "p33":50.1 , "p15":69.3, "p23":93.4, "p17":52.8, "p32":30.3, "p28":46.5, "p31":63.7, "p13":51.3, "p12":39.8, "p11":68.1565}

image reconstruction

In [3]:
def export_nii(np_array, output_path:str, pixel:float=None, header=None):
    if pixel==None:
        pixel_size= ((simind.get_metadata(header))[0].dr)[0]*10 
    else:
        pixel_size= pixel

    affine_matrix= np.array([[pixel_size, 0,  0,  0],
                            [0,  pixel_size, 0,  0],
                            [0,  0,  pixel_size, 0],
                            [0,  0,  0,  1]])
    
    if torch.is_tensor(np_array): 
        np_array= np_array.cpu().numpy()
    
    try:   
        image= np.rot90(np_array, k=2)  
        image = np.flip(image, axis=2)
        nifti_file = nib.Nifti1Image(image, affine=affine_matrix)
        nib.save(nifti_file, output_path)
        print("Volume saved in:", output_path)

    except Exception as e:
        print(f"Error saving NIfTI file: {e}")


In [19]:
activity_escalation= 15 #GBq

In [20]:
activity=activity_escalation/7.4
time=15
num_projections=128

In [21]:
patients = true_lk.keys()
patient="p24"
proj_path= f"F:\\champalimaud\\Ac225\\patients_phantoms\\{patient}\\sim_files\\lu177\\{patient}_lu_tot_w2.h00"
lower_path= f"F:\\champalimaud\\Ac225\\patients_phantoms\\{patient}\\sim_files\\lu177\\{patient}_lu_tot_w1.h00"
hct_path= f"F:\\champalimaud\\Ac225\\patients_phantoms\\{patient}\\sim_files\\lu177\\{patient}_lu.hct"

In [22]:
ct= functions.ct(hct_path, num_slices=128, position=1, cmap=None)

In [23]:
proj, real_proj= functions.projections(proj_path, num_projections, time=time, activity=activity, cmap="magma")

interactive(children=(IntSlider(value=0, continuous_update=False, description='slice_p', max=127), Output()), …

In [24]:
scatter= functions.scatter_estimation(proj_path, lower_path, upper_path=None, weight= 0.475, time=time, activity= activity)
recon= functions.recon(real_proj, ct, proj_path, iterations= 5, subsets=8, scatter= scatter, psf_path=None)

psf correction with 2d gaussian


In [27]:
calib_factor= 1/0.065/(num_projections*time)
calib_volume= recon * calib_factor
output_volume= f"F:\\champalimaud\\tese_artigo\\{patient}\\{activity_escalation}GBq\\{patient}_24h.nii"
export_nii(calib_volume, output_volume, header= proj_path)

Volume saved in: F:\champalimaud\tese_artigo\p24\15GBq\p24_24h.nii


In [3]:
def quantification(volume: np.array, seg: np.array):
    dict_results= {}
    for label in [2,3]:
      mask = seg == label
      nvox = mask.sum()
      mean = volume[mask].mean()
      total = mean * nvox
      dict_results[label]=total
    return dict_results

In [ ]:
adj=["p11", "p32", "p28"]
lu_seg= ["p12", "p28"]

patient="p28"
image_path= f"F:\\champalimaud\\Ac225\\patients_phantoms\\{patient}\\recon_volumes_1\\lu177\\{patient}_lu177.nii"
seg_path= f"F:\\champalimaud\\Ac225\\patients_phantoms\\{patient}\\recon_volumes\\seg_spect_24h_w_tumours_resampled.nrrd"
if patient in adj: 
	seg_path= f"F:\\champalimaud\\Ac225\\patients_phantoms\\{patient}\\recon_volumes\\seg_spect_24h_w_tumours_resampled_1.nrrd"
elif patient in lu_seg:
	seg_path= f"F:\\champalimaud\\Ac225\\patients_phantoms\\{patient}\\recon_volumes\\seg_spect_24h_w_tumours_resampled_lu.nrrd"
seg_arr= sitk.GetArrayFromImage(functions.read_dicom(seg_path))

image_arr= sitk.GetArrayFromImage(functions.read_dicom(image_path))
dict_results_patient= quantification(image_arr, seg_arr)
value_rk, value_lk= dict_results_patient[2], dict_results_patient[3]

true_rk_qt, true_lk_qt= true_rk[patient], true_lk[patient]
            
error_rk= (value_rk-true_rk_qt)/true_rk_qt*100
error_lk=(value_lk-true_lk_qt)/true_lk_qt*100


print(f"right kidney: {value_rk :.3f}, error(%): {error_rk :.4f}")
print(f"left kidney: {value_lk :.3f}, error(%): {error_lk :.4f}")

right kidney: 43.306, error(%): -6.8690
left kidney: 41.412, error(%): -13.3650


In [5]:
def quantification(volume: np.array, seg: np.array):
    dict_results= {}
    for label in [1,5]:
      mask = seg == label
      nvox = mask.sum()
      mean = volume[mask].mean()
      total = mean * nvox
      dict_results[label]=total
    return dict_results

In [7]:
adj=["p12", "p11", "p32"]
patient="p28"
image_path= f"F:\\champalimaud\\Ac225\\patients_phantoms\\{patient}\\recon_volumes_1\\lu177\\{patient}_lu177.nii"
seg_path= f"F:\\champalimaud\\Ac225\\patients_phantoms\\{patient}\\recon_volumes\\seg_spect_24h_w_tumours_resampled.nrrd"
if patient in adj: 
	seg_path= f"F:\\champalimaud\\Ac225\\patients_phantoms\\{patient}\\recon_volumes\\seg_spect_24h_w_tumours_resampled_1.nrrd"
seg_arr= sitk.GetArrayFromImage(functions.read_dicom(seg_path))

image_arr= sitk.GetArrayFromImage(functions.read_dicom(image_path))
dict_results_patient= quantification(image_arr, seg_arr)
value_rk, value_lk= dict_results_patient[1], dict_results_patient[5]

true_rk_qt, true_lk_qt= true_rk[patient], true_lk[patient]
            
error_rk= (value_rk-true_rk_qt)/true_rk_qt*100
error_lk=(value_lk-true_lk_qt)/true_lk_qt*100


print(f"spleen: {value_rk :.3f}")
print(f"liver: {value_lk :.3f}")

spleen: 15.805
liver: 192.705


In [ ]:
patients= ["p15", "p24", "p32", "p33", "p17", "p23", "p25",  "p31", "p13", "p12", "p28"]
for patient in patients:
	image_path= f"F:\\champalimaud\\tese_artigo\\{patient}\\{activity_escalation}GBq\\{patient}_24h.nii"
	img= functions.read_dicom(image_path)
	arr= sitk.GetArrayFromImage(img)
	print(np.sum(arr))

3610.0654
1449.9039
1101.9531
1697.0023
3002.8242
2009.7365
1973.682
1996.2025
1587.6915
1068.2235
2287.6636
